# 🎓 Predicting Student Academic Success and Exam Outcomes Using Supervised Machine Learning

**Project Synopsis Submission**  
* **Author:** Asmita Chandram Kumbar  
* **Class:** SYBSc CS (2nd Year)  
* **Roll No:** SI1161  
* **Domain:** Educational Data Mining & Machine Learning Classification  

---
## 1. Problem Statement
> *“Development of an automated classification model to predict whether a student will pass or fail an examination based on key behavioral metrics, enabling early intervention for at-risk learners.”*

## 2. Project Objectives
1. **Data Preparation**: Clean and preprocess student behavioral dataset, handle missing values, and scale numerical features.
2. **Model Training**: Implement Supervised ML algorithms (Logistic Regression, Decision Tree Classifier, and Random Forest).
3. **Model Evaluation**: Benchmark accuracy, precision, recall, F1-score, confusion matrix, and ROC-AUC curves.
4. **Inference & Decision Support**: Deliver actionable predictive insights to recommend timely academic interventions.

### Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

# Scikit-Learn Modules
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Visualization setup
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.sans-serif': 'Arial', 'font.size': 11})
print("Libraries successfully imported!")

### Step 2: Load and Inspect Dataset

In [ ]:
# Load the student performance dataset
df = pd.read_csv('data/student_data.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Check data summary and missing values
print("--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Target Distribution (Pass / Fail) ---")
print(df['passed'].value_counts(normalize=True))

### Step 3: Exploratory Data Analysis (EDA)
Analyzing trends, correlations, and behavioral patterns influencing student outcomes.

In [ ]:
# 1. Attendance vs Outcome
plt.figure(figsize=(7, 5))
sns.boxplot(x='passed', y='attendance_rate', data=df, hue='passed', palette=['#e74c3c', '#2ecc71'], legend=False)
plt.title('Attendance Rate by Exam Outcome (Pass vs Fail)', weight='bold')
plt.xlabel('Exam Outcome')
plt.ylabel('Attendance Rate (%)')
plt.xticks([0, 1], ['Fail (0)', 'Pass (1)'])
plt.show()

In [ ]:
# 2. Study Hours vs Exam Score
plt.figure(figsize=(8, 5))
sns.regplot(x='study_hours_per_week', y='final_exam_score', data=df, 
            scatter_kws={'alpha': 0.4, 'color': '#3498db'}, line_kws={'color': '#e74c3c'})
plt.title('Weekly Study Hours vs Final Exam Score', weight='bold')
plt.xlabel('Weekly Study Hours')
plt.ylabel('Final Exam Score')
plt.show()

In [ ]:
# 3. Correlation Heatmap
plt.figure(figsize=(9, 7))
num_cols = ['attendance_rate', 'study_hours_per_week', 'previous_scores', 'assignment_completion_rate', 'tutoring_sessions', 'sleep_hours', 'extracurricular', 'internet_access', 'passed']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', mask=mask)
plt.title('Feature Correlation Matrix', weight='bold')
plt.show()

### Step 4: Data Preprocessing & Splitting (80/20 Train-Test Split)

In [ ]:
feature_cols = [
    'attendance_rate', 'study_hours_per_week', 'previous_scores',
    'assignment_completion_rate', 'tutoring_sessions', 'sleep_hours',
    'extracurricular', 'internet_access'
]
target_col = 'passed'

X = df[feature_cols].copy()
y = df[target_col].copy()

# 80% Train, 20% Test with Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Imputation of missing values (median strategy)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

print(f"Train size: {X_train_scaled.shape[0]}, Test size: {X_test_scaled.shape[0]}")

### Step 5: Supervised Model Training & Benchmarking
Training **Logistic Regression**, **Decision Tree**, and **Random Forest**.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5, min_samples_split=10),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6)
}

eval_summary = []

for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    probs = clf.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    
    eval_summary.append({
        'Algorithm': name,
        'Accuracy': f"{acc*100:.2f}%",
        'Precision': f"{prec*100:.2f}%",
        'Recall': f"{rec*100:.2f}%",
        'F1-Score': f"{f1*100:.2f}%",
        'ROC-AUC': f"{roc_auc:.4f}"
    })
    
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))

summary_table = pd.DataFrame(eval_summary)
summary_table

### Step 6: Visualizing Model Comparisons (Confusion Matrix & ROC Curves)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, clf) in enumerate(models.items()):
    y_pred = clf.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Fail', 'Pass'], yticklabels=['Fail', 'Pass'])
    axes[i].set_title(f"{name}", weight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
for name, clf in models.items():
    probs = clf.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")
    
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC = 0.50)')
plt.title('ROC Curves Comparison', weight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

### Step 7: Logistic Regression Feature Weights (Odds Drivers)

In [ ]:
log_reg = models['Logistic Regression']
weights_df = pd.DataFrame({
    'Feature': feature_cols,
    'Weight_Coefficient': log_reg.coef_[0],
    'Odds_Ratio': np.exp(log_reg.coef_[0])
}).sort_values(by='Weight_Coefficient', ascending=False)

weights_df

### Step 8: Interactive Prediction Function

In [ ]:
def predict_student_outcome(attendance, study_hours, prev_scores, assignments, 
                            tutoring=2, sleep=7.0, extracurricular=1, internet=1):
    """
    Predicts student pass/fail outcome and provides timely intervention advice.
    """
    sample = pd.DataFrame([[attendance, study_hours, prev_scores, assignments, tutoring, sleep, extracurricular, internet]],
                          columns=feature_cols)
    sample_scaled = scaler.transform(imputer.transform(sample))
    
    pred = log_reg.predict(sample_scaled)[0]
    prob = log_reg.predict_proba(sample_scaled)[0][1] * 100
    
    print(f"\n--- Prediction Report for Student ---")
    print(f"Pass Likelihood: {prob:.2f}%")
    if pred == 1:
        print("Result: PREDICTED TO PASS ✅")
        print("Guidance: Maintain consistent study routine and attendance.")
    else:
        print("Result: AT-RISK: PREDICTED TO FAIL ⚠️")
        print("Timely Interventions Recommended:")
        if attendance < 75:
            print(f"  - Critical Attendance Deficit: {attendance}% (< 75% statutory limit).")
        if study_hours < 12:
            print(f"  - Increase study hours by at least +4 hrs/week (Current: {study_hours} hrs).")
        if assignments < 75:
            print(f"  - Urgent: Submit pending assignment coursework.")

# Test Case 1: Diligent Student
predict_student_outcome(attendance=88.0, study_hours=20.0, prev_scores=78.0, assignments=95.0)

# Test Case 2: At-Risk Struggling Student
predict_student_outcome(attendance=55.0, study_hours=5.0, prev_scores=42.0, assignments=45.0)